# Does an Inverted Yield Curve Reliably Predict Recessions?

**Aidan Braccia — ECON 630, Assignment 6**

Question, data sources, and method are laid out in `Plan.md`. This notebook
follows that plan stage by stage.

## Stage 1 — Data pull, validation, and cleaning

Everything is pulled live from FRED on each run, so no CSVs are stored in the
repo. The validation asserts run immediately after the pull, before any
analysis, so a bad API response fails loudly instead of quietly corrupting
downstream results.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from fredapi import Fred

load_dotenv()                                   # reads FRED_API_KEY from .env (gitignored)
assert os.getenv("FRED_API_KEY"), "FRED_API_KEY not found - is .env present?"

fred = Fred(api_key=os.getenv("FRED_API_KEY"))  # key is never typed into the notebook

### Pull the three series

Each `get_series` call returns a plain `pd.Series` with a `DatetimeIndex` and no
column name — one value per month.

In [ ]:
gs10  = fred.get_series("GS10")   # 10-Year Treasury Constant Maturity Rate, monthly, percent
gs2   = fred.get_series("GS2")    # 2-Year  Treasury Constant Maturity Rate, monthly, percent
usrec = fred.get_series("USREC")  # NBER recession indicator, monthly, 1 = recession

for name, s in [("GS10", gs10), ("GS2", gs2), ("USREC", usrec)]:
    print(f"{name:6} {len(s):5} observations   {s.index.min():%Y-%m} to {s.index.max():%Y-%m}")

### Validation checks on the raw pull

These are the data-validation unit tests from `Plan.md`. They catch a scrambled
or malformed API response before anything is derived from it — if `USREC` ever
came back with a value other than 0 or 1, every recession count downstream would
be silently wrong.

In [ ]:
for name, s in [("GS10", gs10), ("GS2", gs2), ("USREC", usrec)]:
    assert s.index.is_unique, f"{name}: duplicate dates in the index"
    assert s.index.is_monotonic_increasing, f"{name}: dates are not in ascending order"

assert usrec.dropna().isin([0, 1]).all(), "USREC contains values other than 0/1"

print("raw-pull checks passed")

### Trim to the sample period and build the spread

Sample starts 1990-01, covering four NBER recessions (1990–91, 2001, 2007–09,
2020) plus the 2022–24 inversion episode.

`.loc[START:]` keeps every month from January 1990 onward — it is a slice from
that date to the end of the series, not a match on that single date.

The sign is the thing to get right: `spread = GS10 - GS2`, so an inversion is
`spread < 0`. Flipping it would silently invert the entire story.

In [ ]:
START = "1990-01-01"
gs10  = gs10.loc[START:]
gs2   = gs2.loc[START:]
usrec = usrec.loc[START:]

spread = gs10 - gs2   # 10y minus 2y, so inversion is spread < 0

print(f"sample: {spread.index.min():%Y-%m} to {spread.index.max():%Y-%m}  ({len(spread)} months)")

### Validation checks after cleaning

The alignment asserts confirm no rows were lost when the series were combined.
The spot checks confirm the sign and units are right by testing the spread in
months when the curve is known to have been inverted.

**One assert differs from `Plan.md`.** The plan spot-checked
`spread.loc["2007-09"] < 0`, but that month is **+0.51**. The inversion ahead of
the 2007–09 recession ran from 2006-02 to 2007-05, so by September 2007 the Fed
had begun cutting and the curve had re-steepened. The check moves to **2006-11**
(−0.14), the deepest month of that same inversion — same test, corrected date.

A second spot check on **2023-07** (−0.93) is added for the 2022–24 episode.

In [ ]:
assert gs10.index.equals(gs2.index), "GS10 and GS2 indexes do not match"
assert len(spread) == len(usrec),    "spread and USREC differ in length - rows lost in alignment"
assert spread.index.equals(usrec.index), "spread and USREC are not aligned date-for-date"

expected = pd.date_range(spread.index.min(), spread.index.max(), freq="MS")
assert spread.index.equals(expected), "gaps or extra dates in the monthly index"

assert spread.notna().all(), "spread has missing values - check for gaps in GS10/GS2"

assert spread.loc["2006-11"].item() < 0, \
    "expected the curve to be inverted in 2006-11, ahead of the 2007-2009 recession"
assert spread.loc["2023-07"].item() < 0, \
    "expected the curve to be inverted in 2023-07, deepest month of the 2022-24 episode"

assert spread.between(-5, 5).all(), "spread outside plausible range - check units"

print("cleaning checks passed")
print(f"spread range: {spread.min():.2f} to {spread.max():.2f} percentage points")

---

**Stop here.** Per `NOTES.md`, review this stage before building the charts:
confirm the FRED calls are real (not hardcoded numbers), that the series IDs are
right, that no `.fillna()` / `.dropna()` snuck in as a silent decision, and that
you can say what each assert would catch if it failed. Then log the stage in
`NOTES.md` — including whether the corrected spot-check date should be written
back into `Plan.md`.